# Notebook 3 — CNN + TCN (Temporal Convolutional Network) for Parkinson's Detection

**Model:** CNN + TCN for EEG-based neurological disorder detection.

- **Disorder:** Parkinson's Disease (PD)
- **Dataset:** OpenNeuro ds004584 (Resting-state EEG)
- **Input:** Preprocessed EEG epochs shaped `(N, 60, 500)` — 60 channels, 2s @ 250Hz
- **Output:** Binary classification — PD (1) vs Control (0)
- **Data split:** Subject-wise Train/Val/Test (70/15/15)

> Uses preprocessed data from `data/processed/ds004584_cnn_tcn/`

In [ ]:
# ===== Imports =====
import os
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# ===== Config =====
@dataclass
class Config:
    # Data paths (relative to this notebook in models/ folder)
    data_dir: str = "../data/processed/ds004584_cnn_tcn"

    # Model parameters (will be updated after loading data)
    n_channels: int = 60       # 60 common EEG channels
    n_classes: int = 2         # Binary: PD (1) vs Control (0)
    sampling_rate: int = 250   # From preprocessing
    epoch_seconds: float = 2.0
    n_samples: int = 500       # 2s × 250Hz

    # Training hyperparameters
    batch_size: int = 32
    lr: float = 5e-4
    weight_decay: float = 5e-4
    epochs: int = 50
    patience: int = 15         # Early stopping patience
    dropout: float = 0.5
    label_smoothing: float = 0.1

    # Data augmentation
    use_augmentation: bool = True
    noise_std: float = 0.15
    time_mask_ratio: float = 0.15
    channel_drop_prob: float = 0.1
    time_shift_max: int = 25   # samples; 25 @ 250Hz = 100ms

    # Mixup
    use_mixup: bool = True
    mixup_alpha: float = 0.2

cfg = Config()
print(f"Data directory: {cfg.data_dir}")
print(f"Channels: {cfg.n_channels}, Samples: {cfg.n_samples}")
print(f"Batch size: {cfg.batch_size}, LR: {cfg.lr}, Epochs: {cfg.epochs}")

## Data Loading

Loading preprocessed subject-wise split data from `data/processed/ds004584_cnn_tcn/`:
- `train.npz`, `val.npz`, `test.npz` — Subject-wise split data
- Each file: `X` (N, 60, 500), `y` (N,), `subject_id` (N,)
- Already z-score normalized (stats from training set only)

In [ ]:
# ===== Load preprocessed data =====
def load_split(data_dir: str, split: str):
    """Load a data split (train/val/test) from .npz file."""
    path = Path(data_dir) / f"{split}.npz"
    if not path.exists():
        raise FileNotFoundError(
            f"Expected preprocessed data at {path}.\n"
            "Run preprocessing/preprocess_ds004584_cnn_tcn.py first."
        )
    data = np.load(path, allow_pickle=True)
    X = data["X"].astype(np.float32)   # (N, C, T)
    y = data["y"].astype(np.int64)     # (N,)
    subject_id = data["subject_id"] if "subject_id" in data.files else None
    return X, y, subject_id

# Load all splits
print("Loading preprocessed data...")
X_train, y_train, sid_train = load_split(cfg.data_dir, "train")
X_val, y_val, sid_val = load_split(cfg.data_dir, "val")
X_test, y_test, sid_test = load_split(cfg.data_dir, "test")

# Update config with actual dimensions
cfg.n_channels = X_train.shape[1]
cfg.n_samples = X_train.shape[2]

print(f"\n{'='*50}")
print("DATASET LOADED")
print(f"{'='*50}")
print(f"Train: {X_train.shape} | PD: {(y_train==1).sum()}, Control: {(y_train==0).sum()}")
print(f"Val:   {X_val.shape} | PD: {(y_val==1).sum()}, Control: {(y_val==0).sum()}")
print(f"Test:  {X_test.shape} | PD: {(y_test==1).sum()}, Control: {(y_test==0).sum()}")
print(f"\nChannels: {cfg.n_channels}, Samples: {cfg.n_samples}")

In [ ]:
# ===== EEG Data Augmentation =====
class EEGAugmentation:
    """EEG-specific data augmentation (training only)."""
    def __init__(self, noise_std=0.1, time_mask_ratio=0.1, channel_drop_prob=0.0, time_shift_max=0):
        self.noise_std = noise_std
        self.time_mask_ratio = time_mask_ratio
        self.channel_drop_prob = channel_drop_prob
        self.time_shift_max = time_shift_max

    def add_gaussian_noise(self, x):
        return x + torch.randn_like(x) * self.noise_std

    def time_masking(self, x):
        T = x.shape[-1]
        mask_len = int(T * self.time_mask_ratio)
        if mask_len > 0:
            start = torch.randint(0, T - mask_len, (1,)).item()
            x[..., start:start + mask_len] = 0
        return x

    def channel_dropout(self, x):
        if self.channel_drop_prob <= 0:
            return x
        C = x.shape[-2]
        drop_mask = torch.rand(C, device=x.device) < self.channel_drop_prob
        if drop_mask.any():
            x = x.clone()
            x[drop_mask, :] = 0
        return x

    def time_shift(self, x):
        if self.time_shift_max <= 0:
            return x
        shift = int(torch.randint(-self.time_shift_max, self.time_shift_max + 1, (1,)).item())
        if shift == 0:
            return x
        return torch.roll(x, shifts=shift, dims=-1)

    def __call__(self, x):
        if torch.rand(1).item() < 0.5:
            x = self.add_gaussian_noise(x)
        if torch.rand(1).item() < 0.5:
            x = self.time_masking(x.clone())
        if torch.rand(1).item() < 0.5:
            x = self.channel_dropout(x)
        if torch.rand(1).item() < 0.5:
            x = self.time_shift(x)
        return x


# ===== Torch Dataset =====
class EEGDataset(Dataset):
    def __init__(self, X, y, augment=False, aug_config=None):
        self.X = torch.from_numpy(X)   # (N, C, T)
        self.y = torch.from_numpy(y)   # (N,)
        self.augment = augment
        self.augmenter = EEGAugmentation(
            noise_std=aug_config.noise_std if aug_config else 0.1,
            time_mask_ratio=aug_config.time_mask_ratio if aug_config else 0.1,
            channel_drop_prob=aug_config.channel_drop_prob if aug_config else 0.0,
            time_shift_max=aug_config.time_shift_max if aug_config else 0
        ) if augment else None

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment and self.augmenter:
            x = self.augmenter(x)
        return x, self.y[idx]


# Create data loaders
train_dataset = EEGDataset(X_train, y_train, augment=cfg.use_augmentation, aug_config=cfg)
val_dataset = EEGDataset(X_val, y_val, augment=False)
test_dataset = EEGDataset(X_test, y_test, augment=False)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, drop_last=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

## Model: CNN + TCN (Temporal Convolutional Network)

**Architecture:**
1. **CNN front-end** — Temporal conv (1×25) → Spatial conv (C×1) extracts spatiotemporal features
2. **TCN backbone** — 3 dilated causal TCN blocks (dilation 1, 2, 4) capture multi-scale temporal patterns
3. **Classifier** — Global average pooling → Linear layer for class logits + confidence head

**Advantages:** Lightweight, no recurrence, parallelizable, effective for EEG temporal dynamics

In [ ]:
# ===== CNN + TCN Model =====
class TCNBlock(nn.Module):
    """Temporal Convolutional Network block with residual connection."""
    def __init__(self, in_ch, out_ch, k=3, dilation=1, dropout=0.2):
        super().__init__()
        pad = (k - 1) * dilation  # causal padding
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=k, dilation=dilation, padding=pad)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=k, dilation=dilation, padding=pad)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.down = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        # x: (B, F, T)
        y = self.drop(self.act(self.bn1(self.conv1(x))))
        y = self.drop(self.act(self.bn2(self.conv2(y))))
        # Trim extra padding to match input length (causal padding produces extra samples)
        if y.shape[-1] != x.shape[-1]:
            y = y[..., :x.shape[-1]]
        return self.act(y + self.down(x))


class CNN_TCN(nn.Module):
    """CNN + Temporal Convolutional Network for EEG classification."""
    def __init__(self, n_channels, n_classes, T, dropout=0.4):
        super().__init__()
        # 1) CNN front-end: temporal filtering + spatial filtering
        self.front = nn.Sequential(
            # Temporal convolution: (B,1,C,T) -> (B,16,C,T)
            nn.Conv2d(1, 16, kernel_size=(1, 25), padding=(0, 12), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            # Spatial convolution: (B,16,C,T) -> (B,32,1,T)
            nn.Conv2d(16, 32, kernel_size=(n_channels, 1), bias=False),
            nn.BatchNorm2d(32),
            nn.ELU(),
        )

        # 2) Pooling + dropout
        self.pool = nn.AvgPool2d(kernel_size=(1, 2))  # T -> T//2
        self.drop2d = nn.Dropout(dropout)

        # 3) TCN backbone: multi-scale temporal modeling
        self.tcn = nn.Sequential(
            TCNBlock(32, 32, k=3, dilation=1, dropout=dropout / 2),
            TCNBlock(32, 32, k=3, dilation=2, dropout=dropout / 2),
            TCNBlock(32, 32, k=3, dilation=4, dropout=dropout / 2),
        )

        # 4) Classifier head
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(32, n_classes)
        self.conf_head = nn.Linear(32, 1)  # Confidence head

    def forward(self, x, return_features=False):
        x = x.unsqueeze(1)           # (B, C, T) -> (B, 1, C, T)
        x = self.front(x)            # (B, 32, 1, T)
        x = self.pool(x)             # (B, 32, 1, T//2)
        x = self.drop2d(x)
        x = x.squeeze(2)             # (B, 32, T//2)
        x = self.tcn(x)              # (B, 32, T//2)
        feats = self.gap(x).squeeze(-1)  # (B, 32)
        logits = self.classifier(feats)
        conf = torch.sigmoid(self.conf_head(feats))
        if return_features:
            return logits, conf, feats
        return logits


# Instantiate model
T = cfg.n_samples
model = CNN_TCN(cfg.n_channels, cfg.n_classes, T=T, dropout=cfg.dropout).to(DEVICE)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# ===== Training & Evaluation Utilities =====
def mixup_data(x, y, alpha=0.0):
    """Apply Mixup to a batch."""
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def train_one_epoch(model, loader, optimizer, criterion, mixup_alpha=0.0):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()

        if mixup_alpha > 0:
            xb, y_a, y_b, lam = mixup_data(xb, yb, mixup_alpha)
            logits = model(xb)
            loss = lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
        else:
            logits = model(xb)
            loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)


def eval_model(model, loader):
    """Evaluate model on a data loader."""
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            y_true.append(yb.numpy())
            y_pred.append(pred)
            y_probs.append(probs.cpu().numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    y_probs = np.concatenate(y_probs)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='binary')
    cm = confusion_matrix(y_true, y_pred)
    return acc, f1, cm, y_true, y_pred, y_probs

## Training Loop

- Uses subject-wise **Train/Val/Test** split (no cross-validation leakage)
- Early stopping based on validation F1 score
- Saves best model checkpoint
- Class-weighted loss to handle PD/Control imbalance

In [ ]:
# ===== Training with Early Stopping =====

# Compute class weights to handle imbalance
n_control = (y_train == 0).sum()
n_pd = (y_train == 1).sum()
weight_control = len(y_train) / (2.0 * n_control)
weight_pd = len(y_train) / (2.0 * n_pd)
class_weights = torch.FloatTensor([weight_control, weight_pd]).to(DEVICE)
print(f"Class weights: Control={weight_control:.3f}, PD={weight_pd:.3f}")

# Re-instantiate fresh model
model = CNN_TCN(cfg.n_channels, cfg.n_classes, T=cfg.n_samples, dropout=cfg.dropout).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=cfg.label_smoothing)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=7)

best_val_f1 = -1
best_state = None
patience_counter = 0

print(f"\nTraining CNN-TCN for {cfg.epochs} epochs (patience={cfg.patience})...")
print(f"{'='*70}")

for epoch in range(1, cfg.epochs + 1):
    # Train
    mixup_alpha = cfg.mixup_alpha if cfg.use_mixup else 0.0
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, mixup_alpha=mixup_alpha)

    # Validate
    val_acc, val_f1, val_cm, _, _, _ = eval_model(model, val_loader)

    # Learning rate scheduling
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_f1)
    new_lr = optimizer.param_groups[0]['lr']

    # Early stopping check
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        marker = " *** best ***"
    else:
        patience_counter += 1
        marker = ""

    lr_info = f" (lr reduced to {new_lr:.1e})" if new_lr < old_lr else ""
    if epoch % 2 == 0 or epoch == 1 or marker or lr_info:
        print(f"Epoch {epoch:03d} | loss={train_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}{marker}{lr_info}")

    if patience_counter >= cfg.patience:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {cfg.patience} epochs)")
        break

# Load best model
model.load_state_dict(best_state)
model = model.to(DEVICE)
print(f"\nBest validation F1: {best_val_f1:.4f}")

# Save best model
save_dir = "../training/saved_models"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "cnn_tcn_parkinsons.pth")
torch.save(best_state, save_path)
print(f"Model saved to: {save_path}")

## Test Set Evaluation

Final evaluation on the held-out test set (unseen subjects).
Reports accuracy, F1-score, confusion matrix, and per-class metrics.

In [ ]:
# ===== Test Set Evaluation =====
print("=" * 70)
print("TEST SET EVALUATION (CNN-TCN)")
print("=" * 70)

test_acc, test_f1, test_cm, test_y_true, test_y_pred, test_y_probs = eval_model(model, test_loader)

print(f"\nTest Accuracy:  {test_acc:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")
print(f"\nConfusion Matrix:")
print(f"                Predicted")
print(f"                Control   PD")
print(f"  Actual Control  {test_cm[0,0]:5d}  {test_cm[0,1]:5d}")
print(f"  Actual PD       {test_cm[1,0]:5d}  {test_cm[1,1]:5d}")

# Per-class metrics
print(f"\n{'='*70}")
print("CLASSIFICATION REPORT")
print(f"{'='*70}")
print(classification_report(
    test_y_true, test_y_pred,
    target_names=["Control (0)", "PD (1)"],
    digits=4
))

# Sensitivity and Specificity
tn, fp, fn, tp = test_cm.ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  # PD recall
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # Control recall
print(f"Sensitivity (PD Recall):      {sensitivity:.4f}")
print(f"Specificity (Control Recall): {specificity:.4f}")
print(f"{'='*70}")